In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
import numpy as np
from torch.utils.data import DataLoader, Subset
import torch
from tqdm import tqdm
from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score,
    precision_score, f1_score
)

from dataset.dataset import Dataset
from utils.config import load_config
from utils.builder import build_trainer
from utils.metrics import accuracy, sensitivity, specificity, roc_auc

In [2]:
config_path = "../configs/fcn_full.yaml"

In [3]:
config = load_config(config_path)
data_path = "../"+config["data"]["data_path"]

torch.manual_seed(config["experiment"]["seed"])
np.random.seed(config["experiment"]["seed"])

dataset = Dataset(data_path, "G", config["data"]["batch_size"], use_tabular=config["data"]["use_tabular"], tabular_features=config["data"]["tabular_features"])
train_loader, val_loader, test_loader = dataset.get_loaders()

dataset_f = Dataset(data_path, "F", config["data"]["batch_size"])
_, val_loader_f, test_loader_f = dataset_f.get_loaders()

dataset_m = Dataset(data_path, "M", config["data"]["batch_size"])       
_, val_loader_m, test_loader_m = dataset_m.get_loaders()

In [4]:
trainer = build_trainer(config)
trainer.load_checkpoint()

In [5]:
def predict(model, loader):
    all_predicted = []
    model.eval()
    for inputs, targets in loader:
        with torch.no_grad():
            outputs = model(inputs.to(next(model.parameters()).device))
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_predicted.extend(probs)
    return all_predicted

In [6]:
preds_f = predict(trainer.model, test_loader_f)
preds_m = predict(trainer.model, test_loader_m)

/home/maryna/anaconda3/envs/compmed/lib/python3.9/site-packages/torch/nn/modules/conv.py:366: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1027.)
  return F.conv1d(


In [8]:
from scipy.stats import mannwhitneyu

stat, p = mannwhitneyu(preds_f, preds_m, alternative='two-sided')

if p < 0.05:
    print("Bias detected")
else:
    print("No bias detected")

No bias detected
